# Column Level Security (CLS) — `vstone_catalog.security`

| Column | Visible to | Others get |
|---|---|---|
| `total_market_value_usd` | `admin_group`, `finance_group` | `**REDACTED**` |
| `avg_price_usd` | `admin_group`, `finance_group` | `**REDACTED**` |
| `price_rub` | `admin_group`, `finance_group` | `**REDACTED**` |
| `price_usd` | `admin_group`, `finance_group` | `**REDACTED**` |
| `price_per_hp_usd` | `admin_group`, `finance_group` | `**REDACTED**` |
| `color_r/g/b` | `admin_group` only | `NULL` |


## Step 1 — Inspect base Gold tables

In [0]:
SELECT * FROM vstone_catalog.gold.agg_top_10_brands_by_spend
ORDER BY total_market_value_usd DESC;

## Step 2 — Confirm user identity and group membership

In [0]:
SELECT
  CURRENT_USER()              AS current_user,
  IS_MEMBER('finance_group')  AS is_finance,
  IS_MEMBER('admin_group')    AS is_admin;

## Step 3 — CLS view on `agg_top_10_brands_by_spend`

Revenue and price columns are `**REDACTED**` for non-finance, non-admin users.

In [0]:
-- ============================================================
-- CLS VIEW: agg_top_10_brands_by_spend
-- IS_MEMBER() used -- works with workspace-local groups (Free Edition).
-- CAST to STRING required because REDACTED string must match column output type.
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.cls_brand_market_data AS
SELECT
  brand,
  total_listings,

  -- Sensitive: total revenue
  CAST(
    CASE
      WHEN IS_MEMBER('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(total_market_value_usd AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS total_market_value_usd,

  -- Sensitive: average price
  CAST(
    CASE
      WHEN IS_MEMBER('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(avg_price_usd AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS avg_price_usd,

  gold_load_dt
FROM vstone_catalog.gold.agg_top_10_brands_by_spend;

## Step 4 — CLS view on `fact_listings`

In [0]:
-- ============================================================
-- CLS VIEW: fact_listings
-- Dim joins resolve string labels from integer surrogate keys.
-- Sensitive columns masked via CASE WHEN IS_MEMBER().
-- ============================================================
CREATE OR REPLACE VIEW vstone_catalog.security.cls_fact_listings AS
SELECT
  -- Always visible (non-sensitive)
  f.listing_id,
  dc.brand,
  dc.model,
  f.manufacture_year,
  f.listing_date,
  dp.price_category,
  f.mileage_km,
  dc.fuel_type,
  dl.city_name,
  f.photo_count,
  f.car_age_at_listing,
  f.is_high_mileage,

  -- Sensitive: raw prices -- REDACTED for non-finance
  CAST(
    CASE
      WHEN IS_MEMBER('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(f.price_rub AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS price_rub,

  CAST(
    CASE
      WHEN IS_MEMBER('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(f.price_usd AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS price_usd,

  CAST(
    CASE
      WHEN IS_MEMBER('admin_group') OR IS_MEMBER('finance_group')
        THEN CAST(f.price_per_hp_usd AS STRING)
      ELSE '**REDACTED**'
    END
  AS STRING) AS price_per_hp_usd,

  -- Sensitive: RGB color -- NULL for non-admin
  CASE WHEN IS_MEMBER('admin_group') THEN f.color_r ELSE NULL END AS color_r,
  CASE WHEN IS_MEMBER('admin_group') THEN f.color_g ELSE NULL END AS color_g,
  CASE WHEN IS_MEMBER('admin_group') THEN f.color_b ELSE NULL END AS color_b,

  -- Audit always visible
  f.silver_load_dt,
  f.gold_load_dt

FROM vstone_catalog.gold.fact_listings f
LEFT JOIN vstone_catalog.gold.dim_car            dc ON f.car_sk             = dc.car_sk
                                                   AND dc.__END_AT IS NULL
LEFT JOIN vstone_catalog.gold.dim_price_category dp ON f.price_category_key = dp.price_category_key
LEFT JOIN vstone_catalog.gold.dim_location       dl ON f.location_sk        = dl.location_sk
                                                   AND dl.__END_AT IS NULL;

## Step 5 — Verify masked output

In [0]:
-- finance_group / admin_group : sees real prices
-- others                      : sees **REDACTED**
SELECT * FROM vstone_catalog.security.cls_brand_market_data
ORDER BY total_listings DESC;

SELECT listing_id, brand, price_rub, price_usd, price_per_hp_usd, price_category, color_r
FROM vstone_catalog.security.cls_fact_listings
LIMIT 3;